# 04 — Model & RAG Evaluation

End-to-end evaluation of:
1. FinBERT sentiment classifier (accuracy, macro-F1, per-class metrics)
2. RAG pipeline retrieval quality (precision@k, source diversity)
3. Latency benchmarking for production readiness
4. Business impact summary

In [ ]:
import sys, time, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
print("Libraries loaded.")

## Part 1 — FinBERT Classification Evaluation

In [ ]:
from src.sentiment.model import FinBERTSentiment
from src.sentiment.evaluator import evaluate_predictions, plot_confusion_matrix
from sklearn.preprocessing import LabelEncoder

# 200-sample evaluation set
dataset = load_dataset('takala/financial_phrasebank', 'sentences_allagree', trust_remote_code=True)
df = dataset['train'].to_pandas()
label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
df['sentiment'] = df['label'].map(label_map)

eval_df = df.sample(200, random_state=42).reset_index(drop=True)

model = FinBERTSentiment()
results = model.predict(eval_df['sentence'].tolist())

le = LabelEncoder().fit(['negative', 'neutral', 'positive'])
y_true = le.transform(eval_df['sentiment'].tolist()).tolist()
y_pred = le.transform([r.label for r in results]).tolist()

metrics = evaluate_predictions(y_true, y_pred)

print("=== FinBERT Evaluation (ProsusAI/finbert baseline) ===")
print(f"Accuracy  : {metrics['accuracy']:.4f}")
print(f"Macro-F1  : {metrics['f1_macro']:.4f}")
print(f"Weighted-F1: {metrics['f1_weighted']:.4f}")
print()
print("Per-class F1:")
for label, m in metrics['per_class'].items():
    print(f"  {label:10s} | P={m['precision']:.3f}  R={m['recall']:.3f}  F1={m['f1']:.3f}  n={int(m['support'])}")

In [ ]:
plot_confusion_matrix(
    y_true, y_pred,
    save_path='../outputs/figures/04_confusion_matrix.png'
)

## Part 2 — Confidence Calibration

A well-calibrated model has confidence ≈ accuracy: if the model says 90% confident, it should be right ~90% of the time.

In [ ]:
from collections import defaultdict

# Bin predictions by confidence and compute accuracy per bin
bins = np.linspace(0.3, 1.0, 8)
bin_stats = defaultdict(lambda: {'total': 0, 'correct': 0, 'conf': []})

for res, true_label in zip(results, eval_df['sentiment'].tolist()):
    bin_idx = int(np.digitize(res.score, bins)) - 1
    bin_idx = max(0, min(bin_idx, len(bins) - 2))
    bin_stats[bin_idx]['total'] += 1
    bin_stats[bin_idx]['conf'].append(res.score)
    if res.label == true_label:
        bin_stats[bin_idx]['correct'] += 1

bin_labels, acc_per_bin, mean_conf = [], [], []
for i in range(len(bins) - 1):
    if bin_stats[i]['total'] > 0:
        bin_labels.append(f'{bins[i]:.2f}-{bins[i+1]:.2f}')
        acc_per_bin.append(bin_stats[i]['correct'] / bin_stats[i]['total'])
        mean_conf.append(np.mean(bin_stats[i]['conf']))

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(bin_labels))
ax.bar(x, acc_per_bin, alpha=0.7, label='Accuracy', color='#0f766e')
ax.plot(x, mean_conf, 'o--', color='#7c3aed', label='Mean confidence', lw=2)
ax.axhline(0.5, color='#dc2626', ls=':', alpha=0.5, label='Random baseline')
ax.set_xticks(x)
ax.set_xticklabels(bin_labels, rotation=15)
ax.set_xlabel('Confidence Bin')
ax.set_ylabel('Accuracy / Confidence')
ax.set_title('Calibration Plot — Confidence vs Accuracy')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/04_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

ece = sum(abs(a - c) * (bin_stats[i]['total'] / len(results))
          for i, (a, c) in enumerate(zip(acc_per_bin, mean_conf)))
print(f"Expected Calibration Error (ECE): {ece:.4f}  (lower = better, <0.05 is good)")

## Part 3 — Latency Benchmarking

In [ ]:
import time

batch_sizes = [1, 5, 10, 25, 50]
texts_pool = eval_df['sentence'].tolist()

latencies = {}
for bs in batch_sizes:
    batch = (texts_pool * (bs // len(texts_pool) + 1))[:bs]
    times = []
    for _ in range(5):
        t0 = time.perf_counter()
        _ = model.predict(batch)
        times.append(time.perf_counter() - t0)
    latencies[bs] = {
        'total_ms': np.mean(times) * 1000,
        'per_item_ms': np.mean(times) * 1000 / bs,
    }

df_lat = pd.DataFrame(latencies).T
df_lat.index.name = 'batch_size'
print("Latency benchmarks (5-run average):")
print(df_lat.round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(df_lat.index, df_lat['total_ms'], 'o-', color='#7c3aed', lw=2)
axes[0].set_xlabel('Batch Size')
axes[0].set_ylabel('Total Latency (ms)')
axes[0].set_title('Total Batch Latency')

axes[1].plot(df_lat.index, df_lat['per_item_ms'], 's-', color='#0f766e', lw=2)
axes[1].set_xlabel('Batch Size')
axes[1].set_ylabel('Latency per Item (ms)')
axes[1].set_title('Per-Item Latency (batch efficiency)')

plt.tight_layout()
plt.savefig('../outputs/figures/04_latency_benchmarks.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 4 — RAG Retrieval Quality

In [ ]:
from src.rag.retriever import RegulatoryRAG

rag = RegulatoryRAG()

# Annotated question → expected source
eval_questions = [
    ("What is the minimum CET1 capital ratio?",         "Basel Iii Capital Requirements"),
    ("What are LCR requirements for liquidity?",         "Basel Iii Capital Requirements"),
    ("What does the FCA Consumer Duty require?",         "Fca Conduct Rules"),
    ("What are the Principles for Business?",            "Fca Conduct Rules"),
    ("When must Enhanced Due Diligence be performed?",   "Aml Kyc Requirements"),
    ("How long must AML records be retained?",           "Aml Kyc Requirements"),
]

hits_at_1, hits_at_4 = 0, 0
for q, expected_source in eval_questions:
    response = rag.ask(q)
    sources = response.sources

    hit1 = expected_source in sources[:1] if sources else False
    hit4 = expected_source in sources[:4] if sources else False
    hits_at_1 += hit1
    hits_at_4 += hit4

    status_1 = "✓" if hit1 else "✗"
    status_4 = "✓" if hit4 else "✗"
    print(f"[P@1:{status_1}] [P@4:{status_4}] {q[:55]}")

print()
print(f"Precision@1: {hits_at_1}/{len(eval_questions)} = {hits_at_1/len(eval_questions):.2%}")
print(f"Precision@4: {hits_at_4}/{len(eval_questions)} = {hits_at_4/len(eval_questions):.2%}")

## Part 5 — System Summary & Portfolio Highlights

In [ ]:
print("=" * 65)
print("  FINBERT RAG FINANCIAL INTELLIGENCE — EVALUATION SUMMARY")
print("=" * 65)
print()
print("SENTIMENT ANALYSIS (FinBERT baseline on Financial PhraseBank)")
print(f"  Accuracy     : {metrics['accuracy']:.1%}")
print(f"  Macro-F1     : {metrics['f1_macro']:.4f}")
print(f"  Weighted-F1  : {metrics['f1_weighted']:.4f}")
print()
print("RAG RETRIEVAL")
print(f"  Documents    : 3 regulatory documents (Basel III, FCA, AML/KYC)")
print(f"  Vector store : 28 chunks @ 384 dims (sentence-transformers)")
print(f"  Precision@1  : {hits_at_1/len(eval_questions):.0%}  (out of 6 annotated queries)")
print(f"  Precision@4  : {hits_at_4/len(eval_questions):.0%}")
print()
print("LATENCY (CPU inference)")
print(f"  Single text  : {latencies[1]['total_ms']:.1f} ms")
print(f"  Batch of 10  : {latencies[10]['total_ms']:.1f} ms  ({latencies[10]['per_item_ms']:.1f} ms/item)")
print(f"  Batch of 50  : {latencies[50]['total_ms']:.1f} ms  ({latencies[50]['per_item_ms']:.1f} ms/item)")
print()
print("PORTFOLIO VALUE")
print("  ✓ Domain-adapted NLP (financial sentiment)")
print("  ✓ Production RAG with regulatory grounding")
print("  ✓ Retrieval-only mode — runs offline, no API key needed")
print("  ✓ FastAPI serving + Streamlit demo UI")
print("  ✓ Extensible to any LangChain-compatible LLM (OpenAI, Anthropic, local)")
print("=" * 65)